In [1]:
import argparse
import json
import re
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import numpy as np

In [2]:
# ---- Action mapping ----
def canonical_action(action: str) -> str:
    """Map concrete action tokens to the base keys used in execution_times."""
    if action.startswith("Move"):
        return "MoveTo"
    if action.startswith("PickGoalObj"):
        return "PickGoalObj"
    if action.startswith("PlaceGoalObj"):
        return "PlaceGoalObj"
    return action  # fallback

# ---- Atom parsing & feature extraction ----
_PRED_PAT = re.compile(r"^([A-Za-z_]+)\((.*)\)$")

def _parse_atom(atom: str) -> Tuple[str, List[str]]:
    m = _PRED_PAT.match(atom.strip())
    if not m:
        return atom, []
    pred = m.group(1)
    args = [a.strip() for a in m.group(2).split(",")]
    return pred, args

def summarize_atoms(atoms: List[str]) -> Dict[str, Any]:
    """Cheap, deployable features from the current state's atoms."""
    tables_count: Dict[str, int] = {}
    robot_at: Optional[str] = None
    goalobj_at: Optional[str] = None

    n_clear = 0
    n_on = 0
    n_on_table = 0
    n_blocks_total = 0

    for atom in atoms:
        pred, args = _parse_atom(atom)

        if pred == "RobotAt" and args:
            robot_at = args[0].split(":")[0]

        elif pred == "GoalObjAt" and len(args) >= 2:
            tab = args[1].split(":")[0]
            goalobj_at = tab
            tables_count[tab] = tables_count.get(tab, 0) + 1

        elif pred == "OnTable" and len(args) >= 2:
            n_on_table += 1
            tab = args[1].split(":")[0]
            tables_count[tab] = tables_count.get(tab, 0) + 1

        elif pred == "BlockAt" and len(args) >= 2:
            tab = args[1].split(":")[0]
            tables_count[tab] = tables_count.get(tab, 0) + 1

        elif pred == "On":
            n_on += 1

        elif pred == "Clear":
            n_clear += 1

        for a in args:
            if a.endswith(":block"):
                n_blocks_total += 1

    out = dict(
        robot_table=robot_at,
        goalobj_table=goalobj_at,
        n_tables=len(tables_count),
        n_blocks_total=n_blocks_total,
        n_clear=n_clear,
        n_on=n_on,
        n_on_table=n_on_table,
        blocks_by_table_max=max(tables_count.values()) if tables_count else 0,
        blocks_by_table_sum=sum(tables_count.values()) if tables_count else 0,
    )
    # include per-table counts
    for t, c in tables_count.items():
        out[f"blocks_on_{t}"] = c
    return out

# ---- execution_times normalization ----
def parse_execution_times(exec_times: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    """
    Normalize execution_times into:
      base_action -> {"label_time": float, "priv_feat": Any}
    Expected:
      MoveTo: [euclid_distance(float), time(float)]
      PickGoalObj: [[joint_dists...], time(float)]
    """
    out: Dict[str, Dict[str, Any]] = {}
    for k, v in exec_times.items():
        label = np.nan
        priv = None
        if isinstance(v, list) and len(v) == 2:
            priv = v[0]
            try:
                label = float(v[1])
            except Exception:
                label = np.nan
        else:
            try:
                label = float(v)
            except Exception:
                label = np.nan
        out[k] = {"label_time": label, "priv_feat": priv}
    return out

# ---- one file -> rows ----
def extract_rows_from_json(data: Dict[str, Any], file_name: str, use_all_skeletons: bool=False) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    yss = data.get("yielded_skeletons", [])
    if not yss:
        return rows

    # process either the first, or all skeletons
    iter_skeletons = yss if use_all_skeletons else [yss[0]]

    exec_payload = parse_execution_times(data.get("execution_times", {}))

    for ys in iter_skeletons:
        skeleton: List[str] = ys.get("skeleton", [])
        atoms_seq: List[List[str]] = ys.get("atoms_sequence") or ys.get("atom_sequences") or []

        for step_idx, action in enumerate(skeleton):
            base = canonical_action(action)
            atoms = atoms_seq[step_idx] if step_idx < len(atoms_seq) else []
            feats = summarize_atoms(atoms)
            payload = exec_payload.get(base, {"label_time": np.nan, "priv_feat": None})

            label_time = payload["label_time"]
            priv = payload["priv_feat"]

            move_euclid = np.nan
            pick_joint_sum = np.nan
            pick_joint_len = 0

            if base == "MoveTo":
                if isinstance(priv, (int, float)):
                    move_euclid = float(priv)

            elif base == "PickGoalObj":
                if isinstance(priv, list):
                    try:
                        arr = [float(x) for x in priv]
                        pick_joint_sum = float(np.sum(arr))
                        pick_joint_len = len(arr)
                    except Exception:
                        pass

            row = {
                "file": file_name,
                "step_idx": step_idx,
                "action_token": action,
                "base_action": base,
                "time_sec": label_time,
                "move_euclid_dist": move_euclid,            # deployable for Move
                "pick_joint_norms_sum": pick_joint_sum,     # privileged for Pick
                "pick_joint_norms_len": pick_joint_len,     # privileged for Pick
                "n_atoms": len(atoms),
                "atoms_json": json.dumps(atoms, ensure_ascii=False),
            }
            row.update(feats)
            rows.append(row)

    return rows

# ---- directory -> dataframe ----
def process_directory(input_dir: Path, pattern: str="task_metrics_*.json", max_table_cols: int=16,
                      use_all_skeletons: bool=False) -> pd.DataFrame:
    paths = sorted(input_dir.rglob(pattern))
    all_rows: List[Dict[str, Any]] = []
    for p in paths:
        try:
            data = json.loads(p.read_text())
            rows = extract_rows_from_json(data, p.name, use_all_skeletons=use_all_skeletons)
            all_rows.extend(rows)
        except Exception as e:
            print(f"[WARN] {p}: {e}", file=sys.stderr)

    if not all_rows:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)

    # normalize per-table columns
    table_cols = sorted([c for c in df.columns if c.startswith("blocks_on_")])
    if max_table_cols > 0 and len(table_cols) > max_table_cols:
        # keep top-N by frequency; fold others into 'blocks_on_other'
        col_freq = (df[table_cols] > 0).sum().sort_values(ascending=False)
        keep = list(col_freq.index[:max_table_cols])
        drop = [c for c in table_cols if c not in keep]
        df["blocks_on_other"] = df[drop].sum(axis=1)
        df = df.drop(columns=drop)
        table_cols = keep + ["blocks_on_other"]

    for c in table_cols:
        df[c] = df[c].fillna(0).astype(int)

    front = [
        "file","step_idx","action_token","base_action","time_sec",
        "move_euclid_dist","pick_joint_norms_sum","pick_joint_norms_len",
        "robot_table","goalobj_table",
        "n_tables","n_blocks_total","n_clear","n_on","n_on_table","blocks_by_table_max","blocks_by_table_sum",
        "n_atoms",
    ]
    ordered = [c for c in front if c in df.columns] + table_cols + ["atoms_json"]
    return df[ordered]


In [3]:
INPUT_DIR = Path("/home/cloaked04/projects/predicators/predicators/experiments_pratyush/combined_framework_exp/Execution_time/distance_no_random/search_metrics")
OUTPUT_CSV = Path("/home/cloaked04/projects/predicators/predicators/experiments_pratyush/combined_framework_exp/Execution_time/distance_no_random/actions_dataset.csv") 
OUTPUT_PARQUET = None  # or Path("/path/to/actions_dataset.parquet")

df = process_directory(INPUT_DIR, pattern="task_metrics_*.json",
                       max_table_cols=16, use_all_skeletons=False)

print(df.shape)
display(df.head())

# Save CSV
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)
print("Wrote CSV:", OUTPUT_CSV)

# Optional Parquet
if OUTPUT_PARQUET is not None:
    try:
        OUTPUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(OUTPUT_PARQUET, index=False)  # requires pyarrow or fastparquet
        print("Wrote Parquet:", OUTPUT_PARQUET)
    except Exception as e:
        print("[WARN] Parquet not written:", e)


(222, 21)


,file,step_idx,action_token,base_action,time_sec,move_euclid_dist,pick_joint_norms_sum,pick_joint_norms_len,robot_table,goalobj_table,...,n_blocks_total,n_clear,n_on,n_on_table,blocks_by_table_max,blocks_by_table_sum,n_atoms,blocks_on_table1,blocks_on_table2,atoms_json
0,task_metrics_18415.990956707.json,0,MoveFromHome,MoveTo,9.999432,2.999830,NaN,0,None,table2,...,34,4,7,3,15,16,33,15,1,"[""BlockAt(block1_2_3:block, table1:table)"", ""C..."
1,task_metrics_18415.990956707.json,1,PickGoalObjFromTable,PickGoalObj,39.548557,NaN,27.232491,9,robby,table2,...,34,4,7,3,15,16,33,15,1,"[""BlockAt(block1_2_3:block, table1:table)"", ""C..."
2,task_metrics_20313.972236531.json,0,MoveFromHome,MoveTo,6.388997,1.916699,NaN,0,None,table2,...,33,3,8,2,14,15,33,14,1,"[""OnTableGoalObj(goalObj1_0_0:goal, table1:tab..."
3,task_metrics_20313.972236531.json,1,PickGoalObjFromTable,PickGoalObj,2.904206,NaN,4.997493,9,robby,table2,...,33,3,8,2,14,15,33,14,1,"[""OnTableGoalObj(goalObj1_0_0:goal, table1:tab..."
4,task_metrics_21212.203063822.json,0,MoveFromHome,MoveTo,6.381529,1.914459,NaN,0,None,table2,...,33,3,8,2,14,15,33,14,1,"[""OnTableGoalObj(goalObj1_0_0:goal, table1:tab..."


Wrote CSV: /home/cloaked04/projects/predicators/predicators/experiments_pratyush/combined_framework_exp/Execution_time/distance_no_random/actions_dataset.csv


In [12]:
%cd /home/cloaked04/projects/predicators/predicators/experiments_pratyush/combined_framework_exp/

/home/cloaked04/projects/predicators/predicators/experiments_pratyush/combined_framework_exp


In [13]:
%ls

EE_final_pos_spread.ipynb       ee_cache.json
Execution_time/                 execution_time_distance.py
Untitled.ipynb                  execution_time_model.ipynb
Untitled1.ipynb                 experiments_combined_motion_planning.py
Untitled2.ipynb                 multitable_full_tamp_run.py
cluttered_table.jsonl           pick_execution_time_test.py
cluttered_table1.jsonl          stall_restart_watchdog.py
debug_scratch                   test_multitable_env.ipynb
diff_drive_pick_place.py        test_nb.ipynb
diff_drive_test.ipynb           test_nb1.ipynb
disjoint_diff_drive_test.ipynb  uncluttered_table.jsonl


In [15]:
pd.read_csv("/home/cloaked04/projects/predicators/predicators/experiments_pratyush/combined_framework_exp/Execution_time/distance_no_random/actions_dataset.csv")

,file,step_idx,action_token,base_action,time_sec,move_euclid_dist,pick_joint_norms_sum,pick_joint_norms_len,robot_table,goalobj_table,...,n_blocks_total,n_clear,n_on,n_on_table,blocks_by_table_max,blocks_by_table_sum,n_atoms,blocks_on_table1,blocks_on_table2,atoms_json
0,task_metrics_18415.990956707.json,0,MoveFromHome,MoveTo,9.999432,2.999830,NaN,0,NaN,table2,...,34,4,7,3,15,16,33,15,1,"[""BlockAt(block1_2_3:block, table1:table)"", ""C..."
1,task_metrics_18415.990956707.json,1,PickGoalObjFromTable,PickGoalObj,39.548557,NaN,27.232491,9,robby,table2,...,34,4,7,3,15,16,33,15,1,"[""BlockAt(block1_2_3:block, table1:table)"", ""C..."
2,task_metrics_20313.972236531.json,0,MoveFromHome,MoveTo,6.388997,1.916699,NaN,0,NaN,table2,...,33,3,8,2,14,15,33,14,1,"[""OnTableGoalObj(goalObj1_0_0:goal, table1:tab..."
3,task_metrics_20313.972236531.json,1,PickGoalObjFromTable,PickGoalObj,2.904206,NaN,4.997493,9,robby,table2,...,33,3,8,2,14,15,33,14,1,"[""OnTableGoalObj(goalObj1_0_0:goal, table1:tab..."
4,task_metrics_21212.203063822.json,0,MoveFromHome,MoveTo,6.381529,1.914459,NaN,0,NaN,table2,...,33,3,8,2,14,15,33,14,1,"[""OnTableGoalObj(goalObj1_0_0:goal, table1:tab..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
217,task_metrics_31698.644026898.json,1,PickGoalObjFromTable,PickGoalObj,3.136148,NaN,5.664177,9,robby,table2,...,33,3,8,2,14,15,33,14,1,"[""OnTableGoalObj(goalObj1_0_0:goal, table1:tab..."
218,task_metrics_31726.855192282.json,0,MoveFromHome,MoveTo,4.722576,1.416773,NaN,0,NaN,table2,...,33,3,8,2,14,15,33,14,1,"[""OnTableGoalObj(goalObj1_0_0:goal, table1:tab..."
219,task_metrics_31726.855192282.json,1,PickGoalObjFromTable,PickGoalObj,3.239478,NaN,6.288506,9,robby,table2,...,33,3,8,2,14,15,33,14,1,"[""OnTableGoalObj(goalObj1_0_0:goal, table1:tab..."
220,task_metrics_31754.675029208.json,0,MoveFromHome,MoveTo,4.722576,1.416773,NaN,0,NaN,table2,...,34,4,7,3,15,16,33,15,1,"[""BlockOnGoalObj(block1_0_2:block, goalObj1_0_..."


In [16]:
import json, re, sys, os, joblib
from pathlib import Path
from typing import List, Dict, Any, Tuple

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [17]:
DATA_CSV = Path("/home/cloaked04/projects/predicators/predicators/experiments_pratyush/combined_framework_exp/Execution_time/distance_no_random/actions_dataset.csv")   # ← change if needed
df = pd.read_csv(DATA_CSV)
print(df.shape)
df.head()

(222, 21)


,file,step_idx,action_token,base_action,time_sec,move_euclid_dist,pick_joint_norms_sum,pick_joint_norms_len,robot_table,goalobj_table,...,n_blocks_total,n_clear,n_on,n_on_table,blocks_by_table_max,blocks_by_table_sum,n_atoms,blocks_on_table1,blocks_on_table2,atoms_json
0,task_metrics_18415.990956707.json,0,MoveFromHome,MoveTo,9.999432,2.999830,NaN,0,NaN,table2,...,34,4,7,3,15,16,33,15,1,"[""BlockAt(block1_2_3:block, table1:table)"", ""C..."
1,task_metrics_18415.990956707.json,1,PickGoalObjFromTable,PickGoalObj,39.548557,NaN,27.232491,9,robby,table2,...,34,4,7,3,15,16,33,15,1,"[""BlockAt(block1_2_3:block, table1:table)"", ""C..."
2,task_metrics_20313.972236531.json,0,MoveFromHome,MoveTo,6.388997,1.916699,NaN,0,NaN,table2,...,33,3,8,2,14,15,33,14,1,"[""OnTableGoalObj(goalObj1_0_0:goal, table1:tab..."
3,task_metrics_20313.972236531.json,1,PickGoalObjFromTable,PickGoalObj,2.904206,NaN,4.997493,9,robby,table2,...,33,3,8,2,14,15,33,14,1,"[""OnTableGoalObj(goalObj1_0_0:goal, table1:tab..."
4,task_metrics_21212.203063822.json,0,MoveFromHome,MoveTo,6.381529,1.914459,NaN,0,NaN,table2,...,33,3,8,2,14,15,33,14,1,"[""OnTableGoalObj(goalObj1_0_0:goal, table1:tab..."
